In [ ]:
# ============================================================
# NOTEBOOK 7 — MODEL 2: YOLO26s OBJECT DETECTION
# CELL 1 — GOOGLE DRIVE + IMPORTS
# ============================================================

from google.colab import drive

drive.mount("/content/drive")

import os
import gc
import json
import time
import shutil
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

print("=" * 70)
print("NOTEBOOK 7 — YOLO26s OBJECT DETECTION")
print("=" * 70)

print("\nPyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Image_captioning"
)

assert PROJECT_DIR.exists(), (
    "Image_captioning project folder not found."
)

print("\nProject:")
print(PROJECT_DIR)

print("\n✓ ENVIRONMENT READY")

Mounted at /content/drive
NOTEBOOK 7 — YOLO26s OBJECT DETECTION

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4

Project:
/content/drive/MyDrive/Image_captioning

✓ ENVIRONMENT READY


In [ ]:
# ============================================================
# CELL 2 — INSTALL ULTRALYTICS
# ============================================================

print("=" * 70)
print("INSTALLING / VERIFYING ULTRALYTICS")
print("=" * 70)

!pip install -q -U ultralytics

from ultralytics import YOLO
import ultralytics

print("\nUltralytics version:")
print(ultralytics.__version__)

print("\n✓ ULTRALYTICS READY")

INSTALLING / VERIFYING ULTRALYTICS
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 2.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.

Ultralytics version:
8.4.142

✓ ULTRALYTICS READY


In [ ]:
# ============================================================
# NOTEBOOK 7 — CELL 3A
# RESTORE DATASETS TO LOCAL COLAB STORAGE
# ============================================================

from pathlib import Path
import shutil
import subprocess
import time
import os

print("=" * 70)
print("RESTORING DATASETS TO LOCAL COLAB STORAGE")
print("=" * 70)


# ============================================================
# PROJECT PATH
# ============================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Image_captioning"
)


# ============================================================
# LOCAL ROOT
# ============================================================

LOCAL_ROOT = Path(
    "/content/Image_captioning_Local"
)

LOCAL_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# FLICKR8K PATHS
# ============================================================

DRIVE_FLICKR8K_IMAGES = Path(
    "/content/drive/MyDrive/Image_captioning/"
    "Dataset/Raw/Flickr8K/Images"
)

LOCAL_FLICKR8K_ROOT = Path(
    "/content/Image_captioning_Local/Flickr8K"
)


# ============================================================
# FLICKR30K ZIP PATH
# ============================================================

FLICKR30K_ZIP = Path(
    "/content/drive/MyDrive/Image_captioning/"
    "Dataset/ZIP/flickr30k.zip"
)

LOCAL_FLICKR30K_ROOT = Path(
    "/content/Image_captioning_Local/Flickr30K"
)


# ============================================================
# VERIFY SOURCE FILES
# ============================================================

print("\nSOURCE VERIFICATION")

print("\nFlickr8K Drive images:")
print(DRIVE_FLICKR8K_IMAGES)

print("Exists:")
print(DRIVE_FLICKR8K_IMAGES.exists())


print("\nFlickr30K ZIP:")
print(FLICKR30K_ZIP)

print("Exists:")
print(FLICKR30K_ZIP.exists())


assert DRIVE_FLICKR8K_IMAGES.exists(), (
    "Flickr8K images folder not found in Google Drive."
)

assert FLICKR30K_ZIP.exists(), (
    "Flickr30K ZIP file not found in Google Drive."
)


# ============================================================
# RESTORE FLICKR8K
# ============================================================

print("\n" + "=" * 70)
print("RESTORING FLICKR8K")
print("=" * 70)


if LOCAL_FLICKR8K_ROOT.exists():

    existing_images = list(
        LOCAL_FLICKR8K_ROOT.rglob("*.jpg")
    )

else:

    existing_images = []


print(
    "\nExisting local Flickr8K images:",
    len(existing_images)
)


if len(existing_images) >= 8091:

    print(
        "\n✓ Flickr8K already restored locally"
    )

else:

    print(
        "\nCopying Flickr8K images from Drive "
        "to local Colab storage..."
    )

    if LOCAL_FLICKR8K_ROOT.exists():

        shutil.rmtree(
            LOCAL_FLICKR8K_ROOT
        )


    start_time = time.time()


    shutil.copytree(
        DRIVE_FLICKR8K_IMAGES,
        LOCAL_FLICKR8K_ROOT
    )


    elapsed = time.time() - start_time


    print(
        f"\n✓ Flickr8K restored in "
        f"{elapsed / 60:.2f} minutes"
    )


# ============================================================
# VERIFY FLICKR8K
# ============================================================

flickr8k_images = list(
    LOCAL_FLICKR8K_ROOT.rglob("*.jpg")
)

print(
    "\nLocal Flickr8K images:",
    len(flickr8k_images)
)


assert len(flickr8k_images) >= 8091, (
    f"Expected 8091 Flickr8K images but found "
    f"{len(flickr8k_images)}"
)


print(
    "✓ Flickr8K verification successful"
)


# ============================================================
# RESTORE FLICKR30K
# ============================================================

print("\n" + "=" * 70)
print("RESTORING FLICKR30K")
print("=" * 70)


if LOCAL_FLICKR30K_ROOT.exists():

    existing_30k_images = list(
        LOCAL_FLICKR30K_ROOT.rglob("*.jpg")
    )

else:

    existing_30k_images = []


print(
    "\nExisting local Flickr30K JPG files:",
    len(existing_30k_images)
)


if len(existing_30k_images) >= 31783:

    print(
        "\n✓ Flickr30K already restored locally"
    )

else:

    print(
        "\nExtracting Flickr30K ZIP directly "
        "to local Colab storage..."
    )


    if LOCAL_FLICKR30K_ROOT.exists():

        shutil.rmtree(
            LOCAL_FLICKR30K_ROOT
        )


    LOCAL_FLICKR30K_ROOT.mkdir(
        parents=True,
        exist_ok=True
    )


    start_time = time.time()


    result = subprocess.run(
        [
            "unzip",
            "-q",
            str(FLICKR30K_ZIP),
            "-d",
            str(LOCAL_FLICKR30K_ROOT)
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )


    elapsed = time.time() - start_time


    print(
        f"\nExtraction time: "
        f"{elapsed / 60:.2f} minutes"
    )


    if result.returncode != 0:

        print("\nUNZIP ERROR:")
        print(result.stderr)

        raise RuntimeError(
            "Flickr30K extraction failed."
        )


# ============================================================
# VERIFY FLICKR30K
# ============================================================

flickr30k_images = list(
    LOCAL_FLICKR30K_ROOT.rglob("*.jpg")
)


print(
    "\nLocal Flickr30K JPG files:",
    len(flickr30k_images)
)


assert len(flickr30k_images) >= 31783, (
    f"Expected at least 31,783 Flickr30K images "
    f"but found {len(flickr30k_images)}"
)


print(
    "✓ Flickr30K verification successful"
)


# ============================================================
# FINAL STATUS
# ============================================================

print("\n" + "=" * 70)
print("LOCAL DATASET RESTORATION COMPLETE")
print("=" * 70)


print(
    "\nFlickr8K local root:"
)

print(
    LOCAL_FLICKR8K_ROOT
)


print(
    "\nFlickr8K images:",
    len(flickr8k_images)
)


print(
    "\nFlickr30K local root:"
)

print(
    LOCAL_FLICKR30K_ROOT
)


print(
    "\nFlickr30K JPG files:",
    len(flickr30k_images)
)


print("\n" + "=" * 70)
print("✓ READY FOR YOLO26s")
print("✓ NOW RUN THE ORIGINAL CELL 3 AGAIN")
print("=" * 70)

RESTORING DATASETS TO LOCAL COLAB STORAGE

SOURCE VERIFICATION

Flickr8K Drive images:
/content/drive/MyDrive/Image_captioning/Dataset/Raw/Flickr8K/Images
Exists:
True

Flickr30K ZIP:
/content/drive/MyDrive/Image_captioning/Dataset/ZIP/flickr30k.zip
Exists:
True

RESTORING FLICKR8K

Existing local Flickr8K images: 0

Copying Flickr8K images from Drive to local Colab storage...

✓ Flickr8K restored in 4.21 minutes

Local Flickr8K images: 8091
✓ Flickr8K verification successful

RESTORING FLICKR30K

Existing local Flickr30K JPG files: 0

Extracting Flickr30K ZIP directly to local Colab storage...

Extraction time: 4.68 minutes

Local Flickr30K JPG files: 63566
✓ Flickr30K verification successful

LOCAL DATASET RESTORATION COMPLETE

Flickr8K local root:
/content/Image_captioning_Local/Flickr8K

Flickr8K images: 8091

Flickr30K local root:
/content/Image_captioning_Local/Flickr30K

Flickr30K JPG files: 63566

✓ READY FOR YOLO26s
✓ NOW RUN THE ORIGINAL CELL 3 AGAIN


In [ ]:
# ============================================================
# CELL 3 — PROJECT PATHS
# ============================================================

print("=" * 70)
print("YOLO26s PROJECT PATH CONFIGURATION")
print("=" * 70)


# ------------------------------------------------------------
# INPUT
# ------------------------------------------------------------

CAPTIONS_FILE = Path(
    "/content/drive/MyDrive/Image_captioning/"
    "Dataset/Processed/Captions/"
    "captions_processed.csv"
)


# ------------------------------------------------------------
# LOCAL IMAGE STORAGE
# ------------------------------------------------------------

LOCAL_FLICKR8K_ROOT = Path(
    "/content/Image_captioning_Local/Flickr8K"
)

LOCAL_FLICKR30K_ROOT = Path(
    "/content/Image_captioning_Local/Flickr30K"
)


# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

OUTPUT_ROOT = Path(
    "/content/drive/MyDrive/Image_captioning/"
    "Outputs/YOLO26_Detection"
)

CHECKPOINT_DIR = (
    OUTPUT_ROOT / "Checkpoints"
)

FINAL_DIR = (
    OUTPUT_ROOT / "Final"
)

STATS_DIR = (
    OUTPUT_ROOT / "Statistics"
)


for directory in [
    OUTPUT_ROOT,
    CHECKPOINT_DIR,
    FINAL_DIR,
    STATS_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


print("\nCAPTIONS FILE:")
print(CAPTIONS_FILE)
print("Exists:", CAPTIONS_FILE.exists())

print("\nLOCAL FLICKR8K:")
print(LOCAL_FLICKR8K_ROOT)
print("Exists:", LOCAL_FLICKR8K_ROOT.exists())

print("\nLOCAL FLICKR30K:")
print(LOCAL_FLICKR30K_ROOT)
print("Exists:", LOCAL_FLICKR30K_ROOT.exists())

print("\nOUTPUT ROOT:")
print(OUTPUT_ROOT)


assert CAPTIONS_FILE.exists(), (
    "Processed captions file not found."
)

assert LOCAL_FLICKR8K_ROOT.exists(), (
    "Local Flickr8K directory not found."
)

assert LOCAL_FLICKR30K_ROOT.exists(), (
    "Local Flickr30K directory not found."
)


print("\n✓ ALL REQUIRED PATHS VERIFIED")

YOLO26s PROJECT PATH CONFIGURATION

CAPTIONS FILE:
/content/drive/MyDrive/Image_captioning/Dataset/Processed/Captions/captions_processed.csv
Exists: True

LOCAL FLICKR8K:
/content/Image_captioning_Local/Flickr8K
Exists: True

LOCAL FLICKR30K:
/content/Image_captioning_Local/Flickr30K
Exists: True

OUTPUT ROOT:
/content/drive/MyDrive/Image_captioning/Outputs/YOLO26_Detection

✓ ALL REQUIRED PATHS VERIFIED


In [ ]:
# ============================================================
# CELL 4 — LOAD IMAGE MANIFEST
# ============================================================

print("=" * 70)
print("LOADING IMAGE MANIFEST")
print("=" * 70)


captions_df = pd.read_csv(
    CAPTIONS_FILE
)


print("\nCaption dataframe shape:")
print(captions_df.shape)

print("\nColumns:")
print(
    captions_df.columns.tolist()
)


assert "image_id" in captions_df.columns
assert "dataset" in captions_df.columns
assert "image_name" in captions_df.columns


# ------------------------------------------------------------
# UNIQUE IMAGE MANIFEST
# ------------------------------------------------------------

image_manifest = (
    captions_df[
        [
            "image_id",
            "dataset",
            "image_name"
        ]
    ]
    .drop_duplicates(
        subset=["image_id"]
    )
    .reset_index(
        drop=True
    )
)


print("\nTotal unique images:")
print(
    len(image_manifest)
)


print("\nDataset distribution:")
print(
    image_manifest[
        "dataset"
    ]
    .value_counts()
)


assert len(
    image_manifest
) == 39874


print("\n✓ IMAGE MANIFEST READY")

LOADING IMAGE MANIFEST

Caption dataframe shape:
(199369, 7)

Columns:
['image_id', 'dataset', 'image_name', 'caption', 'processed_caption', 'model_caption', 'caption_length']

Total unique images:
39874

Dataset distribution:
dataset
Flickr30K    31783
Flickr8K      8091
Name: count, dtype: int64

✓ IMAGE MANIFEST READY


In [ ]:
# ============================================================
# CELL 5 — BUILD LOCAL IMAGE PATH MAP
# ============================================================

print("=" * 70)
print("BUILDING LOCAL IMAGE PATH MAP")
print("=" * 70)


# ------------------------------------------------------------
# FLICKR8K
# ------------------------------------------------------------

flickr8k_files = list(
    LOCAL_FLICKR8K_ROOT.rglob("*.jpg")
)

print(
    "\nFlickr8K JPG files:",
    len(flickr8k_files)
)


flickr8k_map = {
    path.name: str(path)
    for path in flickr8k_files
}


# ------------------------------------------------------------
# FLICKR30K
# ------------------------------------------------------------

flickr30k_files = list(
    LOCAL_FLICKR30K_ROOT.rglob("*.jpg")
)

print(
    "\nFlickr30K JPG files found:",
    len(flickr30k_files)
)


# Some duplicate extracted paths can exist.
# Mapping by filename safely gives one path per image.

flickr30k_map = {}

for path in flickr30k_files:

    if path.name not in flickr30k_map:

        flickr30k_map[
            path.name
        ] = str(path)


print(
    "\nFlickr30K unique filenames:",
    len(flickr30k_map)
)


# ------------------------------------------------------------
# RESOLVE IMAGE PATHS
# ------------------------------------------------------------

def resolve_local_path(row):

    dataset = row["dataset"]
    image_name = row["image_name"]


    if dataset == "Flickr8K":

        return flickr8k_map.get(
            image_name
        )


    elif dataset == "Flickr30K":

        return flickr30k_map.get(
            image_name
        )


    return None


image_manifest[
    "local_image_path"
] = image_manifest.apply(
    resolve_local_path,
    axis=1
)


missing_paths = (
    image_manifest[
        "local_image_path"
    ]
    .isna()
    .sum()
)


print(
    "\nImages missing local paths:",
    missing_paths
)


assert missing_paths == 0, (
    "Some images are missing from local storage."
)


print("\nSample paths:")

display(
    image_manifest.head()
)


print("\n✓ ALL IMAGES MAPPED LOCALLY")

BUILDING LOCAL IMAGE PATH MAP

Flickr8K JPG files: 8091

Flickr30K JPG files found: 63566

Flickr30K unique filenames: 31783

Images missing local paths: 0

Sample paths:


,image_id,dataset,image_name,local_image_path
0,Flickr8K/1000268201_693b08cb0e.jpg,Flickr8K,1000268201_693b08cb0e.jpg,/content/Image_captioning_Local/Flickr8K/10002...
1,Flickr8K/1001773457_577c3a7d70.jpg,Flickr8K,1001773457_577c3a7d70.jpg,/content/Image_captioning_Local/Flickr8K/10017...
2,Flickr8K/1002674143_1b742ab4b8.jpg,Flickr8K,1002674143_1b742ab4b8.jpg,/content/Image_captioning_Local/Flickr8K/10026...
3,Flickr8K/1003163366_44323f5815.jpg,Flickr8K,1003163366_44323f5815.jpg,/content/Image_captioning_Local/Flickr8K/10031...
4,Flickr8K/1007129816_e794419615.jpg,Flickr8K,1007129816_e794419615.jpg,/content/Image_captioning_Local/Flickr8K/10071...



✓ ALL IMAGES MAPPED LOCALLY


In [ ]:
# ============================================================
# CELL 6 — RESTORE YOLO26s CHECKPOINT STATE
# ============================================================

print("=" * 70)
print("RESTORING YOLO26s RESUME STATE")
print("=" * 70)


checkpoint_files = sorted(
    CHECKPOINT_DIR.glob(
        "checkpoint_*.csv"
    )
)


print(
    "\nCheckpoint files found:",
    len(checkpoint_files)
)


completed_ids = set()


for checkpoint_file in checkpoint_files:

    checkpoint_df = pd.read_csv(
        checkpoint_file
    )

    if "image_id" in checkpoint_df.columns:

        completed_ids.update(
            checkpoint_df[
                "image_id"
            ]
            .dropna()
            .astype(str)
            .tolist()
        )


print(
    "\nPreviously completed images:",
    len(completed_ids)
)


remaining_manifest = (
    image_manifest[
        ~image_manifest[
            "image_id"
        ].isin(
            completed_ids
        )
    ]
    .reset_index(
        drop=True
    )
)


print(
    "\nTotal dataset images:",
    len(image_manifest)
)

print(
    "\nRemaining images:",
    len(remaining_manifest)
)


# ------------------------------------------------------------
# NEXT CHECKPOINT NUMBER
# ------------------------------------------------------------

if len(checkpoint_files) == 0:

    next_checkpoint_number = 1

else:

    checkpoint_numbers = []

    for checkpoint_file in checkpoint_files:

        try:

            number = int(
                checkpoint_file
                .stem
                .split("_")[-1]
            )

            checkpoint_numbers.append(
                number
            )

        except:
            pass


    next_checkpoint_number = (
        max(checkpoint_numbers) + 1
    )


print(
    "\nNext checkpoint number:",
    next_checkpoint_number
)


print("\n✓ RESUME STATE READY")

RESTORING YOLO26s RESUME STATE

Checkpoint files found: 0

Previously completed images: 0

Total dataset images: 39874

Remaining images: 39874

Next checkpoint number: 1

✓ RESUME STATE READY


In [ ]:
# ============================================================
# CELL 7 — LOAD YOLO26s MODEL
# ============================================================

print("=" * 70)
print("LOADING YOLO26s")
print("=" * 70)


DEVICE = (
    0
    if torch.cuda.is_available()
    else "cpu"
)


print("\nDevice:")
print(DEVICE)


MODEL_NAME = "yolo26s.pt"


yolo26_model = YOLO(
    MODEL_NAME
)


print("\nModel loaded:")
print(MODEL_NAME)


print(
    "\nYOLO26s official pretrained model ready."
)


print("\n✓ YOLO26s READY")

LOADING YOLO26s

Device:
0

Model loaded:
yolo26s.pt

YOLO26s official pretrained model ready.

✓ YOLO26s READY


In [ ]:
# ============================================================
# CELL 8 — YOLO26s SINGLE IMAGE TEST
# ============================================================

print("=" * 70)
print("YOLO26s SINGLE IMAGE TEST")
print("=" * 70)


test_row = image_manifest.iloc[0]

test_image_id = (
    test_row["image_id"]
)

test_image_path = (
    test_row["local_image_path"]
)


print("\nImage ID:")
print(test_image_id)

print("\nImage path:")
print(test_image_path)


results = yolo26_model.predict(
    source=test_image_path,
    conf=0.25,
    imgsz=640,
    device=DEVICE,
    verbose=False
)


result = results[0]


records = []


if result.boxes is not None:

    boxes = (
        result.boxes.xyxy
        .cpu()
        .numpy()
    )

    confidences = (
        result.boxes.conf
        .cpu()
        .numpy()
    )

    classes = (
        result.boxes.cls
        .cpu()
        .numpy()
        .astype(int)
    )


    for box, confidence, cls_id in zip(
        boxes,
        confidences,
        classes
    ):

        records.append(

            {

                "object_name":
                    result.names[
                        int(cls_id)
                    ],

                "confidence":
                    float(confidence),

                "xmin":
                    float(box[0]),

                "ymin":
                    float(box[1]),

                "xmax":
                    float(box[2]),

                "ymax":
                    float(box[3])

            }

        )


test_df = pd.DataFrame(
    records
)


print(
    "\nDetections:",
    len(test_df)
)


display(
    test_df
)


print(
    "\n✓ SINGLE IMAGE TEST COMPLETE"
)

YOLO26s SINGLE IMAGE TEST

Image ID:
Flickr8K/1000268201_693b08cb0e.jpg

Image path:
/content/Image_captioning_Local/Flickr8K/1000268201_693b08cb0e.jpg

Detections: 1


,object_name,confidence,xmin,ymin,xmax,ymax
0,person,0.91443,5.59094,150.679077,106.013069,337.860352



✓ SINGLE IMAGE TEST COMPLETE


In [ ]:
# ============================================================
# CELL 9 — GPU SPEED TEST
# ============================================================

print("=" * 70)
print("YOLO26s SPEED TEST")
print("=" * 70)


TEST_BATCH_SIZE = 64


speed_test_df = (
    remaining_manifest
    .head(
        TEST_BATCH_SIZE
    )
)


if len(speed_test_df) == 0:

    speed_test_df = (
        image_manifest
        .head(
            TEST_BATCH_SIZE
        )
    )


test_paths = (
    speed_test_df[
        "local_image_path"
    ]
    .tolist()
)


if torch.cuda.is_available():

    torch.cuda.empty_cache()

    torch.cuda.synchronize()


start_time = time.time()


speed_results = yolo26_model.predict(
    source=test_paths,
    conf=0.25,
    imgsz=640,
    device=DEVICE,
    verbose=False
)


if torch.cuda.is_available():

    torch.cuda.synchronize()


elapsed = (
    time.time()
    - start_time
)


images_per_second = (
    len(test_paths)
    / elapsed
)


print(
    "\nImages tested:",
    len(test_paths)
)

print(
    "Time:",
    round(elapsed, 2),
    "seconds"
)

print(
    "Speed:",
    round(
        images_per_second,
        2
    ),
    "images/second"
)


print(
    "\n✓ SPEED TEST COMPLETE"
)

YOLO26s SPEED TEST

Images tested: 64
Time: 1.53 seconds
Speed: 41.72 images/second

✓ SPEED TEST COMPLETE


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("❌ GPU is NOT connected")

CUDA available: True
GPU: Tesla T4


In [ ]:
# ============================================================
# CELL 10 — FULL YOLO26s DATASET PROCESSING
# ============================================================

print("=" * 70)
print("YOLO26s FULL DATASET PROCESSING")
print("=" * 70)


# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

BATCH_SIZE = 64

CHECKPOINT_INTERVAL = 512

CONFIDENCE_THRESHOLD = 0.25

IMAGE_SIZE = 640


print("\nConfiguration:")

print("Batch size:", BATCH_SIZE)

print(
    "Checkpoint interval:",
    CHECKPOINT_INTERVAL
)

print(
    "Confidence threshold:",
    CONFIDENCE_THRESHOLD
)

print(
    "Image size:",
    IMAGE_SIZE
)


# ------------------------------------------------------------
# PROCESSING
# ------------------------------------------------------------

batch_records = []

images_since_checkpoint = 0

checkpoint_number = (
    next_checkpoint_number
)

total_processed = 0

start_time = time.time()


for start_index in range(
    0,
    len(remaining_manifest),
    BATCH_SIZE
):

    batch_df = (
        remaining_manifest
        .iloc[
            start_index:
            start_index + BATCH_SIZE
        ]
        .reset_index(
            drop=True
        )
    )


    batch_paths = (
        batch_df[
            "local_image_path"
        ]
        .tolist()
    )


    # --------------------------------------------------------
    # YOLO26s INFERENCE
    # --------------------------------------------------------

    results = yolo26_model.predict(

        source=batch_paths,

        conf=CONFIDENCE_THRESHOLD,

        imgsz=IMAGE_SIZE,

        device=DEVICE,

        verbose=False

    )


    # --------------------------------------------------------
    # PROCESS EACH IMAGE
    # --------------------------------------------------------

    for row, result in zip(

        batch_df.itertuples(
            index=False
        ),

        results

    ):

        image_id = row.image_id

        image_path = row.local_image_path


        # ----------------------------------------------------
        # ZERO DETECTIONS
        # ----------------------------------------------------

        if (
            result.boxes is None
            or len(result.boxes) == 0
        ):

            batch_records.append(

                {

                    "image_id":
                        image_id,

                    "image_path":
                        image_path,

                    "object_name":
                        None,

                    "confidence":
                        None,

                    "xmin":
                        None,

                    "ymin":
                        None,

                    "xmax":
                        None,

                    "ymax":
                        None,

                    "objects_detected":
                        0

                }

            )


        # ----------------------------------------------------
        # DETECTIONS FOUND
        # ----------------------------------------------------

        else:

            boxes = (
                result.boxes.xyxy
                .cpu()
                .numpy()
            )

            confidences = (
                result.boxes.conf
                .cpu()
                .numpy()
            )

            classes = (
                result.boxes.cls
                .cpu()
                .numpy()
                .astype(int)
            )


            detection_count = (
                len(boxes)
            )


            for box, confidence, cls_id in zip(

                boxes,
                confidences,
                classes

            ):

                batch_records.append(

                    {

                        "image_id":
                            image_id,

                        "image_path":
                            image_path,

                        "object_name":
                            result.names[
                                int(cls_id)
                            ],

                        "confidence":
                            float(
                                confidence
                            ),

                        "xmin":
                            float(
                                box[0]
                            ),

                        "ymin":
                            float(
                                box[1]
                            ),

                        "xmax":
                            float(
                                box[2]
                            ),

                        "ymax":
                            float(
                                box[3]
                            ),

                        "objects_detected":
                            detection_count

                    }

                )


    # --------------------------------------------------------
    # PROGRESS
    # --------------------------------------------------------

    processed_in_batch = (
        len(batch_df)
    )

    total_processed += (
        processed_in_batch
    )

    images_since_checkpoint += (
        processed_in_batch
    )


    if (
        total_processed % 512 == 0
        or total_processed
        == len(remaining_manifest)
    ):

        elapsed = (
            time.time()
            - start_time
        )


        speed = (
            total_processed
            / elapsed
        )


        print(

            f"Processed "
            f"{total_processed}/"
            f"{len(remaining_manifest)} "
            f"images | "
            f"{speed:.2f} images/sec"

        )


    # --------------------------------------------------------
    # SAVE CHECKPOINT
    # --------------------------------------------------------

    if (

        images_since_checkpoint
        >= CHECKPOINT_INTERVAL

        or

        total_processed
        == len(remaining_manifest)

    ):


        checkpoint_df = pd.DataFrame(
            batch_records
        )


        checkpoint_path = (
            CHECKPOINT_DIR
            /
            f"checkpoint_"
            f"{checkpoint_number:05d}.csv"
        )


        checkpoint_df.to_csv(

            checkpoint_path,

            index=False

        )


        print(
            f"\n✓ Checkpoint "
            f"{checkpoint_number} saved"
        )

        print(
            "Images processed "
            "in checkpoint:",
            images_since_checkpoint
        )

        print(
            "Records saved:",
            len(checkpoint_df)
        )


        # ----------------------------------------------------
        # RESET
        # ----------------------------------------------------

        batch_records = []

        images_since_checkpoint = 0

        checkpoint_number += 1


        gc.collect()

        if torch.cuda.is_available():

            torch.cuda.empty_cache()


print("\n" + "=" * 70)
print("✓ YOLO26s FULL PROCESSING COMPLETE")
print("=" * 70)

YOLO26s FULL DATASET PROCESSING

Configuration:
Batch size: 64
Checkpoint interval: 512
Confidence threshold: 0.25
Image size: 640
Processed 512/39874 images | 62.34 images/sec

✓ Checkpoint 1 saved
Images processed in checkpoint: 512
Records saved: 2125
Processed 1024/39874 images | 60.96 images/sec

✓ Checkpoint 2 saved
Images processed in checkpoint: 512
Records saved: 1960
Processed 1536/39874 images | 61.40 images/sec

✓ Checkpoint 3 saved
Images processed in checkpoint: 512
Records saved: 2041
Processed 2048/39874 images | 60.68 images/sec

✓ Checkpoint 4 saved
Images processed in checkpoint: 512
Records saved: 2275
Processed 2560/39874 images | 60.93 images/sec

✓ Checkpoint 5 saved
Images processed in checkpoint: 512
Records saved: 1942
Processed 3072/39874 images | 60.30 images/sec

✓ Checkpoint 6 saved
Images processed in checkpoint: 512
Records saved: 2170
Processed 3584/39874 images | 60.05 images/sec

✓ Checkpoint 7 saved
Images processed in checkpoint: 512
Records saved: 

In [ ]:
# ============================================================
# CELL 11 — MERGE YOLO26s CHECKPOINTS
# ============================================================

print("=" * 70)
print("MERGING YOLO26s CHECKPOINTS")
print("=" * 70)


checkpoint_files = sorted(
    CHECKPOINT_DIR.glob(
        "checkpoint_*.csv"
    )
)


print(
    "\nCheckpoint files:",
    len(checkpoint_files)
)


assert len(
    checkpoint_files
) > 0


checkpoint_dfs = []


for checkpoint_file in checkpoint_files:

    print(
        "\nLoading:",
        checkpoint_file.name
    )

    checkpoint_df = pd.read_csv(
        checkpoint_file
    )

    print(
        "Rows:",
        len(checkpoint_df)
    )

    print(
        "Unique images:",
        checkpoint_df[
            "image_id"
        ]
        .nunique()
    )


    checkpoint_dfs.append(
        checkpoint_df
    )


# ------------------------------------------------------------
# COMBINE
# ------------------------------------------------------------

final_df = pd.concat(
    checkpoint_dfs,
    ignore_index=True
)


print(
    "\nTotal records:",
    len(final_df)
)


print(
    "Unique images:",
    final_df[
        "image_id"
    ]
    .nunique()
)


# ------------------------------------------------------------
# REMOVE EXACT DUPLICATES
# ------------------------------------------------------------

before = len(
    final_df
)


final_df = (
    final_df
    .drop_duplicates()
    .reset_index(
        drop=True
    )
)


duplicates_removed = (
    before
    - len(final_df)
)


print(
    "\nDuplicates removed:",
    duplicates_removed
)


# ------------------------------------------------------------
# VERIFY IMAGE COVERAGE
# ------------------------------------------------------------

unique_images = (
    final_df[
        "image_id"
    ]
    .nunique()
)


assert unique_images == 39874, (

    f"Expected 39,874 images "
    f"but found {unique_images}"

)


# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

FINAL_OUTPUT = (
    FINAL_DIR
    /
    "yolo26_detections_full.csv"
)


final_df.to_csv(

    FINAL_OUTPUT,

    index=False

)


print("\nFinal output:")

print(
    FINAL_OUTPUT
)


print(
    "\nFinal shape:",
    final_df.shape
)


print(
    "\n✓ YOLO26s FINAL DETECTION FILE SAVED"
)

MERGING YOLO26s CHECKPOINTS

Checkpoint files: 78

Loading: checkpoint_00001.csv
Rows: 2125
Unique images: 512

Loading: checkpoint_00002.csv
Rows: 1960
Unique images: 512

Loading: checkpoint_00003.csv
Rows: 2041
Unique images: 512

Loading: checkpoint_00004.csv
Rows: 2275
Unique images: 512

Loading: checkpoint_00005.csv
Rows: 1942
Unique images: 512

Loading: checkpoint_00006.csv
Rows: 2170
Unique images: 512

Loading: checkpoint_00007.csv
Rows: 2171
Unique images: 512

Loading: checkpoint_00008.csv
Rows: 2610
Unique images: 512

Loading: checkpoint_00009.csv
Rows: 2501
Unique images: 512

Loading: checkpoint_00010.csv
Rows: 2576
Unique images: 512

Loading: checkpoint_00011.csv
Rows: 2444
Unique images: 512

Loading: checkpoint_00012.csv
Rows: 2543
Unique images: 512

Loading: checkpoint_00013.csv
Rows: 2401
Unique images: 512

Loading: checkpoint_00014.csv
Rows: 2564
Unique images: 512

Loading: checkpoint_00015.csv
Rows: 2074
Unique images: 512

Loading: checkpoint_00016.csv
Rows

In [ ]:
# ============================================================
# CELL 12 — YOLO26s RESEARCH STATISTICS
# ============================================================

print("=" * 70)
print("YOLO26s DETECTION STATISTICS")
print("=" * 70)


# ------------------------------------------------------------
# IMAGE LEVEL COUNTS
# ------------------------------------------------------------

image_counts = (

    final_df

    .groupby(
        "image_id"
    )

    ["objects_detected"]

    .max()

)


total_images = (
    len(image_counts)
)


images_with_detection = (

    image_counts > 0

).sum()


zero_detection_images = (

    image_counts == 0

).sum()


detection_coverage = (

    images_with_detection
    / total_images
    * 100

)


# ------------------------------------------------------------
# DETECTION ROWS
# ------------------------------------------------------------

detected_rows = (

    final_df[
        final_df[
            "objects_detected"
        ] > 0
    ]

)


total_detection_records = (
    len(detected_rows)
)


mean_confidence = (
    detected_rows[
        "confidence"
    ]
    .mean()
)


median_confidence = (
    detected_rows[
        "confidence"
    ]
    .median()
)


unique_classes = (
    detected_rows[
        "object_name"
    ]
    .nunique()
)


mean_objects_per_image = (
    image_counts.mean()
)


median_objects_per_image = (
    image_counts.median()
)


max_objects_per_image = (
    image_counts.max()
)


# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

print(
    "\nTotal images:",
    total_images
)

print(
    "Images with detections:",
    images_with_detection
)

print(
    "Zero-detection images:",
    zero_detection_images
)

print(
    f"Detection coverage: "
    f"{detection_coverage:.4f}%"
)

print(
    "\nTotal detection records:",
    total_detection_records
)

print(
    "Unique object classes:",
    unique_classes
)

print(
    "\nMean confidence:",
    round(
        mean_confidence,
        6
    )
)

print(
    "Median confidence:",
    round(
        median_confidence,
        6
    )
)

print(
    "\nMean objects/image:",
    round(
        mean_objects_per_image,
        4
    )
)

print(
    "Median objects/image:",
    median_objects_per_image
)

print(
    "Maximum objects/image:",
    max_objects_per_image
)


# ------------------------------------------------------------
# SAVE STATISTICS
# ------------------------------------------------------------

stats_df = pd.DataFrame(

    [

        {

            "model":
                "YOLO26s",

            "total_images":
                total_images,

            "images_with_detection":
                images_with_detection,

            "zero_detection_images":
                zero_detection_images,

            "detection_coverage_percent":
                detection_coverage,

            "total_detection_records":
                total_detection_records,

            "unique_object_classes":
                unique_classes,

            "mean_confidence":
                mean_confidence,

            "median_confidence":
                median_confidence,

            "mean_objects_per_image":
                mean_objects_per_image,

            "median_objects_per_image":
                median_objects_per_image,

            "max_objects_per_image":
                max_objects_per_image

        }

    ]

)


STATS_FILE = (

    STATS_DIR
    /
    "yolo26_dataset_statistics.csv"

)


stats_df.to_csv(

    STATS_FILE,

    index=False

)


print("\nStatistics saved:")

print(
    STATS_FILE
)


print(
    "\n✓ RESEARCH STATISTICS COMPLETE"
)

YOLO26s DETECTION STATISTICS

Total images: 39874
Images with detections: 39795
Zero-detection images: 79
Detection coverage: 99.8019%

Total detection records: 238969
Unique object classes: 80

Mean confidence: 0.63267
Median confidence: 0.659171

Mean objects/image: 5.9931
Median objects/image: 4.0
Maximum objects/image: 53

Statistics saved:
/content/drive/MyDrive/Image_captioning/Outputs/YOLO26_Detection/Statistics/yolo26_dataset_statistics.csv

✓ RESEARCH STATISTICS COMPLETE


In [ ]:
# ============================================================
# CELL 13 — FINAL VERIFICATION
# ============================================================

print("=" * 70)
print("YOLO26s FINAL VERIFICATION")
print("=" * 70)


print(
    "\nFinal detection file exists:",
    FINAL_OUTPUT.exists()
)


verification_df = pd.read_csv(
    FINAL_OUTPUT
)


print(
    "\nFinal shape:",
    verification_df.shape
)


print(
    "\nColumns:"
)

print(
    verification_df.columns.tolist()
)


print(
    "\nUnique images:"
)

print(
    verification_df[
        "image_id"
    ]
    .nunique()
)


print(
    "\nImages with detections:"
)

print(

    (
        verification_df
        .groupby(
            "image_id"
        )

        ["objects_detected"]

        .max()

        > 0

    ).sum()

)


print(
    "\nZero detection images:"
)

print(

    (
        verification_df
        .groupby(
            "image_id"
        )

        ["objects_detected"]

        .max()

        == 0

    ).sum()

)


assert (
    verification_df[
        "image_id"
    ]
    .nunique()

    ==

    39874

)


assert (
    len(
        verification_df
    )

    > 0

)


print("\n" + "=" * 70)
print("✓ MODEL 2 — YOLO26s COMPLETE")
print("=" * 70)

print(
    "\nFinal file:"
)

print(
    FINAL_OUTPUT
)

YOLO26s FINAL VERIFICATION

Final detection file exists: True

Final shape: (239048, 9)

Columns:
['image_id', 'image_path', 'object_name', 'confidence', 'xmin', 'ymin', 'xmax', 'ymax', 'objects_detected']

Unique images:
39874

Images with detections:
39795

Zero detection images:
79

✓ MODEL 2 — YOLO26s COMPLETE

Final file:
/content/drive/MyDrive/Image_captioning/Outputs/YOLO26_Detection/Final/yolo26_detections_full.csv
